# 📊 Step 2 — Visualise Combined Dataset

Reads `combined_dataset.csv` and generates one PDF per participant.

- **X-axis**: real clock time, one tick every **5 seconds**
- **Page width**: configurable in minutes (`PAGE_MIN`)
- **3 signal panels**: Nasal Airflow, SpO2, Thoracic Movement
- **Coloured regions**: shaded across all panels for each event
- **Sleep stage**: shown as a colour-coded strip at the top
- **Labels**: event name + sleep stage displayed on each region

**Run `01_build_combined_dataset.ipynb` first.**

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'matplotlib'], check=True)
print('✅ Dependencies ready')

✅ Dependencies ready


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════╗
# ║  CONFIG                                     ║
# ╚══════════════════════════════════════════════╝
BASE_DIR  = Path(r'D:\SRIP_Health')
CSV_PATH  = BASE_DIR / 'Dataset' / 'combined_dataset.csv'
VIZ_DIR   = BASE_DIR / 'Visualizations'
VIZ_DIR.mkdir(exist_ok=True)

PAGE_MIN  = 5          # minutes of recording per PDF page
TICK_SEC  = 5          # x-axis tick every N seconds

# ── Event colours ─────────────────────────────
EVENT_COLORS = {
    'Hypopnea':           '#FF7043',
    'Obstructive Apnea':  '#D32F2F',
    'Central Apnea':      '#7B1FA2',
    'Mixed Apnea':        '#F57F17',
    'Body event':         '#607D8B',
    'Normal':             None,       # no shading for Normal
}
DEFAULT_EVENT_COLOR = '#78909C'

# ── Sleep stage colours (for the stage strip) ─
STAGE_COLORS = {
    'Wake':      '#EF5350',
    'N1':        '#FFA726',
    'N2':        '#42A5F5',
    'N3':        '#1565C0',
    'N4':        '#0D47A1',   # darker blue — deep sleep old terminology
    'REM':       '#66BB6A',
    'Movement':  '#BDBDBD',
    'Unknown':   '#EEEEEE',
}

print(f'Dataset  : {CSV_PATH}')
print(f'Output   : {VIZ_DIR}')
print(f'Page     : {PAGE_MIN} min  |  X-tick every {TICK_SEC} sec')

Dataset  : D:\SRIP_Health\Dataset\combined_dataset.csv
Output   : D:\SRIP_Health\Visualizations
Page     : 5 min  |  X-tick every 5 sec


In [3]:
# ── Load dataset ──────────────────────────────
assert CSV_PATH.exists(), (
    f'❌ {CSV_PATH} not found — run 01_build_combined_dataset.ipynb first'
)

df_all = pd.read_csv(CSV_PATH, parse_dates=['datetime'])
print(f'Loaded {len(df_all):,} rows | columns: {list(df_all.columns)}')
print(f'Participants: {sorted(df_all["participant"].unique())}')
print(f'\nEvent distribution:')
print(df_all['event_label'].value_counts().to_string())
print(f'\nSleep stage distribution:')
print(df_all['sleep_stage'].value_counts().to_string())

Loaded 4,227,209 rows | columns: ['datetime', 'participant', 'sleep_stage', 'event_label', 'nasal', 'spo2', 'thoracic']
Participants: ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']

Event distribution:
event_label
Normal               3710399
Hypopnea              404072
Obstructive Apnea     109428
Body event              2009
Mixed Apnea             1301

Sleep stage distribution:
sleep_stage
Wake        1594688
N2          1172160
N1           633600
N3           511680
REM          312000
Unknown        2400
Movement        681


In [4]:
# ══════════════════════════════════════════════
#  HELPER: extract contiguous event segments
#  from a per-row event_label column
# ══════════════════════════════════════════════

def get_segments(df_page: pd.DataFrame, col: str) -> list[dict]:
    """
    Returns list of {label, start_dt, end_dt} for contiguous
    runs of the same non-Normal label.
    """
    segs = []
    if df_page.empty:
        return segs

    cur_label = None
    cur_start = None

    for _, row in df_page.iterrows():
        lbl = row[col]
        if lbl != cur_label:
            if cur_label is not None and cur_label != 'Normal' and cur_label != 'Unknown':
                segs.append({'label': cur_label,
                             'start': cur_start,
                             'end':   row['datetime']})
            cur_label = lbl
            cur_start = row['datetime']

    # close last segment
    if cur_label is not None and cur_label != 'Normal' and cur_label != 'Unknown':
        segs.append({'label': cur_label,
                     'start': cur_start,
                     'end':   df_page.iloc[-1]['datetime']})
    return segs


def get_stage_segments(df_page: pd.DataFrame) -> list[dict]:
    """Same but for sleep stages (all stages, not just non-Normal)"""
    segs = []
    if df_page.empty:
        return segs
    cur_label = None
    cur_start = None
    for _, row in df_page.iterrows():
        lbl = row['sleep_stage']
        if lbl != cur_label:
            if cur_label is not None:
                segs.append({'label': cur_label,
                             'start': cur_start,
                             'end':   row['datetime']})
            cur_label = lbl
            cur_start = row['datetime']
    if cur_label is not None:
        segs.append({'label': cur_label,
                     'start': cur_start,
                     'end':   df_page.iloc[-1]['datetime']})
    return segs


print('✅ get_segments(), get_stage_segments() ready')

✅ get_segments(), get_stage_segments() ready


In [5]:
# ══════════════════════════════════════════════
#  DRAW ONE PAGE
# ══════════════════════════════════════════════

SIGNALS = [
    ('nasal',    'Nasal Airflow',     '#1565C0'),
    ('spo2',     'SpO₂ (%)',          '#2E7D32'),
    ('thoracic', 'Thoracic Movement', '#E65100'),
]

def draw_page(df_page: pd.DataFrame, pid: str,
              page_num: int, total_pages: int) -> plt.Figure:

    if df_page.empty:
        return None

    t_start = df_page['datetime'].iloc[0]
    t_end   = df_page['datetime'].iloc[-1]

    # ── Figure layout ────────────────────────────
    # Rows: sleep_stage strip (thin) + 3 signals + event timeline strip
    fig, axes = plt.subplots(
        5, 1,
        figsize=(22, 11),
        gridspec_kw={'height_ratios': [0.25, 1, 1, 1, 0.3],
                     'hspace': 0.08},
        sharex=True
    )
    ax_stage = axes[0]
    ax_sig   = axes[1:4]      # nasal, spo2, thoracic
    ax_ev    = axes[4]        # event timeline strip

    # ── Get event and stage segments ─────────────
    ev_segs    = get_segments(df_page, 'event_label')
    stage_segs = get_stage_segments(df_page)

    # Convert datetimes to matplotlib float for axvspan
    def dt2num(dt):
        return matplotlib.dates.date2num(dt)

    # ── Sleep stage strip ────────────────────────
    ax_stage.set_ylim(0, 1)
    ax_stage.set_yticks([])
    ax_stage.spines[['top','right','left','bottom']].set_visible(False)
    for seg in stage_segs:
        col = STAGE_COLORS.get(seg['label'], '#EEEEEE')
        ax_stage.axvspan(dt2num(seg['start']), dt2num(seg['end']),
                         color=col, alpha=0.85, linewidth=0)
        mid = dt2num(seg['start']) + (dt2num(seg['end']) - dt2num(seg['start'])) / 2
        ax_stage.text(mid, 0.5, seg['label'],
                      ha='center', va='center', fontsize=6.5,
                      color='white', fontweight='bold',
                      transform=ax_stage.get_xaxis_transform())
    ax_stage.set_ylabel('Stage', fontsize=7, rotation=0,
                         labelpad=28, va='center')

    # ── Signal panels ────────────────────────────
    for ax, (col, ylabel, color) in zip(ax_sig, SIGNALS):
        ax.plot(df_page['datetime'], df_page[col],
                color=color, linewidth=0.5, alpha=0.9)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(axis='y', labelsize=7)
        ax.grid(axis='y', linestyle=':', alpha=0.4, linewidth=0.5)
        ax.spines[['top','right']].set_visible(False)

        # Shade event regions across all signal panels
        for seg in ev_segs:
            ec = EVENT_COLORS.get(seg['label'], DEFAULT_EVENT_COLOR)
            ax.axvspan(dt2num(seg['start']), dt2num(seg['end']),
                       color=ec, alpha=0.20, linewidth=0)
            # Solid border lines
            ax.axvline(dt2num(seg['start']), color=ec,
                       linewidth=0.9, alpha=0.8, linestyle='--')
            ax.axvline(dt2num(seg['end']),   color=ec,
                       linewidth=0.9, alpha=0.8, linestyle='--')

        # Event label on nasal panel only
        if col == 'nasal':
            y_top = ax.get_ylim()[1]
            for seg in ev_segs:
                ec  = EVENT_COLORS.get(seg['label'], DEFAULT_EVENT_COLOR)
                mid = seg['start'] + (seg['end'] - seg['start']) / 2
                ax.text(dt2num(mid), y_top * 0.92,
                        seg['label'],
                        ha='center', va='top', fontsize=6.5,
                        color='white', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2',
                                  facecolor=ec, edgecolor='none', alpha=0.85))

    # ── Event timeline strip (bottom) ────────────
    ax_ev.set_ylim(0, 1)
    ax_ev.set_yticks([])
    ax_ev.spines[['top','right','left']].set_visible(False)
    ax_ev.set_facecolor('#F5F5F5')
    for seg in ev_segs:
        ec = EVENT_COLORS.get(seg['label'], DEFAULT_EVENT_COLOR)
        ax_ev.axvspan(dt2num(seg['start']), dt2num(seg['end']),
                      color=ec, alpha=0.9, linewidth=0)
        mid = seg['start'] + (seg['end'] - seg['start']) / 2
        ax_ev.text(dt2num(mid), 0.5, seg['label'],
                   ha='center', va='center', fontsize=6,
                   color='white', fontweight='bold')
    ax_ev.set_ylabel('Events', fontsize=7, rotation=0,
                     labelpad=28, va='center')

    # ── X-axis: ticks every TICK_SEC seconds ─────
    loc = matplotlib.dates.SecondLocator(interval=TICK_SEC)
    fmt = matplotlib.dates.DateFormatter('%H:%M:%S')
    ax_ev.xaxis.set_major_locator(loc)
    ax_ev.xaxis.set_major_formatter(fmt)
    plt.setp(ax_ev.xaxis.get_majorticklabels(),
             rotation=45, ha='right', fontsize=6.5)

    # ── Title ────────────────────────────────────
    title = (f'{pid}   '
             f'{t_start.strftime("%Y-%m-%d")}   '
             f'{t_start.strftime("%H:%M:%S")}  to  '
             f'{t_end.strftime("%H:%M:%S")}')
    fig.suptitle(title, fontsize=11, fontweight='bold', y=0.995)

    # ── Legend (events present on this page) ─────
    present_events = {seg['label'] for seg in ev_segs}
    if present_events:
        handles = [
            mpatches.Patch(
                color=EVENT_COLORS.get(e, DEFAULT_EVENT_COLOR),
                label=e, alpha=0.8)
            for e in sorted(present_events)
        ]
        fig.legend(handles=handles,
                   loc='upper right', fontsize=7,
                   framealpha=0.85, ncol=len(handles),
                   bbox_to_anchor=(0.99, 0.99))

    # ── Sleep stage legend ────────────────────────
    present_stages = {seg['label'] for seg in stage_segs}
    stage_handles = [
        mpatches.Patch(color=STAGE_COLORS.get(s,'#EEE'), label=s)
        for s in ['Wake','N1','N2','N3','N4','REM','Movement']
        if s in present_stages
    ]
    if stage_handles:
        fig.legend(handles=stage_handles,
                   loc='upper left', fontsize=7,
                   framealpha=0.85, ncol=len(stage_handles),
                   bbox_to_anchor=(0.01, 0.99))

    # ── Footer ───────────────────────────────────
    fig.text(0.5, 0.005,
             f'Page {page_num} of {total_pages}  |  {pid}',
             ha='center', fontsize=8, color='#555555')

    return fig


print('✅ draw_page() defined')

✅ draw_page() defined


In [6]:
# ══════════════════════════════════════════════
#  GENERATE ONE PDF PER PARTICIPANT
# ══════════════════════════════════════════════

PAGE_SEC  = PAGE_MIN * 60        # seconds per page
PIDS      = sorted(df_all['participant'].unique())

for pid in PIDS:
    print(f'\n{"─"*55}')
    print(f'  Generating PDF for {pid}')

    df_pid = df_all[df_all['participant'] == pid].copy()
    df_pid = df_pid.sort_values('datetime').reset_index(drop=True)

    t_start_rec = df_pid['datetime'].iloc[0]
    t_end_rec   = df_pid['datetime'].iloc[-1]
    total_sec   = (t_end_rec - t_start_rec).total_seconds()
    total_pages = int(np.ceil(total_sec / PAGE_SEC))

    print(f'  Duration  : {total_sec/3600:.2f} h  ({total_pages} pages)')

    out_path = VIZ_DIR / f'{pid}_visualization.pdf'
    with PdfPages(str(out_path)) as pdf:
        for page_i in range(total_pages):
            p_start = t_start_rec + timedelta(seconds=page_i * PAGE_SEC)
            p_end   = t_start_rec + timedelta(seconds=(page_i + 1) * PAGE_SEC)

            df_page = df_pid[
                (df_pid['datetime'] >= p_start) &
                (df_pid['datetime'] <  p_end)
            ]

            if df_page.empty:
                continue

            fig = draw_page(df_page, pid, page_i + 1, total_pages)
            if fig is not None:
                pdf.savefig(fig, bbox_inches='tight', dpi=120)
                plt.close(fig)

            if (page_i + 1) % 10 == 0 or page_i == 0:
                print(f'  Page {page_i+1}/{total_pages}...')

    import os
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'  ✅ Saved → {out_path}  ({size_mb:.1f} MB)')

print(f'\n✅ All PDFs generated in {VIZ_DIR}')


───────────────────────────────────────────────────────
  Generating PDF for AP01
  Duration  : 7.60 h  (92 pages)
  Page 1/92...
  Page 10/92...
  Page 20/92...
  Page 30/92...
  Page 40/92...
  Page 50/92...
  Page 60/92...
  Page 70/92...
  Page 80/92...
  Page 90/92...
  ✅ Saved → D:\SRIP_Health\Visualizations\AP01_visualization.pdf  (4.7 MB)

───────────────────────────────────────────────────────
  Generating PDF for AP02
  Duration  : 7.38 h  (89 pages)
  Page 1/89...
  Page 10/89...
  Page 20/89...
  Page 30/89...
  Page 40/89...
  Page 50/89...
  Page 60/89...
  Page 70/89...
  Page 80/89...
  ✅ Saved → D:\SRIP_Health\Visualizations\AP02_visualization.pdf  (4.0 MB)

───────────────────────────────────────────────────────
  Generating PDF for AP03
  Duration  : 7.07 h  (85 pages)
  Page 1/85...
  Page 10/85...
  Page 20/85...
  Page 30/85...
  Page 40/85...
  Page 50/85...
  Page 60/85...
  Page 70/85...
  Page 80/85...
  ✅ Saved → D:\SRIP_Health\Visualizations\AP03_visualizat